## Unit 2: RAG, Vector Stores, and Indexing

**Part 4a: Embeddings & Vector Space**

In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-huggingface sentence-transformers langchain-community

from dotenv import load_dotenv
load_dotenv()

import os
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



In [2]:
vector = embeddings.embed_query("Orange")

print(f"Dimensionality: {len(vector)}")
print(f"First 5 numbers: {vector[:5]}")

Dimensionality: 384
First 5 numbers: [-0.032363127917051315, 0.026842676103115082, -0.06043584644794464, 0.052247557789087296, 0.041252102702856064]


In [3]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

vec_cat = embeddings.embed_query("Cat")
vec_dog = embeddings.embed_query("Dog")
vec_car = embeddings.embed_query("Car")

print(f"Cat vs Dog: {cosine_similarity(vec_cat, vec_dog):.4f}")
print(f"Cat vs Car: {cosine_similarity(vec_cat, vec_car):.4f}")

Cat vs Dog: 0.6606
Cat vs Car: 0.4633


**Part 4b: Naive RAG Pipeline**

In [4]:
# Setup
%pip install python-dotenv --upgrade --quiet faiss-cpu langchain-huggingface sentence-transformers langchain-community
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Using the same free model as Part 4a
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Enter your Google API Key:  ········


In [5]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Rahul's favorite hobby is playing football on weekends."),
    Document(page_content="The secret password to access the server room is 'Orchid'."),
    Document(page_content="LangChain helps developers build applications using large language models and external data sources."),
]

In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """
Answer based ONLY on the context below:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("What is the secret password?")
print(result)

The secret password to access the server room is 'Orchid'.


**Part 4c: Deep Dive into Indexing Algorithms**

In [8]:
import faiss
import numpy as np

d = 128
nb = 10000
xb = np.random.random((nb, d)).astype('float32')

In [9]:
index = faiss.IndexFlatL2(d)
index.add(xb)
print(f"Flat Index contains {index.ntotal} vectors")

Flat Index contains 10000 vectors


In [10]:
nlist = 100 
quantizer = faiss.IndexFlatL2(d) 
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)

index_ivf.train(xb)
index_ivf.add(xb)

In [11]:
#Printing the firsr five clusters
print("First 5 cluster centroids (showing first 10 values):\n")

for i in range(5):
    centroid = index_ivf.quantizer.reconstruct(i)
    print(f"Cluster {i}: {centroid[:10]}")
    print()

First 5 cluster centroids (showing first 10 values):

Cluster 0: [0.4764727  0.6271634  0.46998098 0.584324   0.5812946  0.4979496
 0.55866164 0.450426   0.43325216 0.5297654 ]

Cluster 1: [0.6291513  0.46006042 0.6251913  0.5009565  0.5289869  0.531896
 0.52094245 0.50156575 0.48131666 0.43975708]

Cluster 2: [0.5890738  0.5473305  0.42602256 0.4719053  0.5045549  0.36022043
 0.43263328 0.5582075  0.57779384 0.60787636]

Cluster 3: [0.33816814 0.46670008 0.44983834 0.5025965  0.4556487  0.5973305
 0.591876   0.6216329  0.5327069  0.45981097]

Cluster 4: [0.547677   0.48753133 0.41895938 0.5444087  0.48777205 0.5201586
 0.46860296 0.46746734 0.50821346 0.59295326]



In [12]:
M = 16 
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.add(xb)

In [13]:
m = 8 
index_pq = faiss.IndexPQ(d, m, 8)
index_pq.train(xb)
index_pq.add(xb)
print("PQ Compression complete. RAM usage minimized.")

PQ Compression complete. RAM usage minimized.
